# Aula 6 — Epigenômica · Notebook da prática

**Disciplina:** Introdução à Bioinformática — PPGED / IME
**Momento 2 da aula (1:30 – 2:35)** · dados públicos e ferramentas de código aberto

Neste notebook você vai:

| Atividade | Tempo | O que faremos | Ferramentas |
|---|---|---|---|
| **P1** | 15 min | Ver a "assinatura" epigenética de um gene ativo e de um reprimido | IGV-Web (no navegador) |
| **P2** | 25 min | Encontrar os pontos do genoma onde a proteína CTCF se liga (ChIP-seq) | MACS3, pandas, matplotlib |
| **P3** | 25 min | Encontrar CpGs com metilação diferente entre dois grupos | pandas, SciPy, statsmodels |

> **Você não precisa saber programar.** Todo o código está pronto. Sua tarefa é **executar, observar e interpretar**. As perguntas marcadas com 🤔 são para você pensar; as respostas esperadas estão escondidas em "Ver resposta".

## 0. Como usar este notebook (leia antes de começar)

Um **notebook** é um documento dividido em **células**:

- **Células de texto** (como esta) explicam o que está acontecendo.
- **Células de código** (fundo cinza) contêm instruções para o computador. Para executar uma, clique nela e aperte o botão **▶** à esquerda (ou `Shift + Enter`). O resultado aparece logo abaixo.

Três convenções que você vai ver no código:

| Símbolo | Significado |
|---|---|
| `# texto` | **Comentário**: explicação para humanos. O computador ignora. |
| `!comando` | **Comando de terminal** (Aula 1), executado de dentro do notebook. |
| `variavel = ...` | Guarda um valor com um nome, para usar depois. |

**Regras de ouro:**
1. Execute as células **em ordem, de cima para baixo**. Uma célula pode depender de algo criado na anterior.
2. Enquanto uma célula roda, aparece um círculo girando no lugar do ▶. Espere terminar.
3. Se algo der errado, use **Ambiente de execução → Reiniciar e executar tudo**.

> **Onde o código roda?** No Google Colab, num computador do Google, não no seu. Nada é instalado na sua máquina.

---
# P1 — Explorando o epigenoma no navegador (15 min)

Esta atividade é feita **fora do notebook**, no **IGV-Web**, um navegador de genomas.

> **O que é o IGV?** O *Integrative Genomics Viewer* (Broad Institute) mostra o genoma como uma régua horizontal e empilha embaixo **trilhas** (*tracks*): genes, sinais de ChIP-seq, picos, metilação. A versão web roda no navegador e já acessa dados do **ENCODE**, o grande consórcio público de dados epigenômicos.

**Os dados:** linhagem celular **GM12878** (células linfoblastoides humanas, derivadas de linfócitos B), uma das mais estudadas do ENCODE.

### Passo a passo
1. Abra <https://igv.org/app> numa nova aba.
2. Menu **Genome** → **Human (GRCh37/hg19)**. *(Usamos hg19 porque os dados da P2 estão nessa versão do genoma.)*
3. Menu **Tracks** → **ENCODE Signals** (o nome pode variar ligeiramente). Na busca, digite `GM12878` e marque as trilhas de sinal:
   - **H3K4me3** → marca de promotor ativo
   - **H3K27ac** → marca de região regulatória ativa
   - **H3K27me3** → marca de repressão (complexo Polycomb)
   - **CTCF** → proteína "isoladora" (vamos usá-la na P2)
   - **DNase** (se disponível) → cromatina aberta
4. Clique em **OK**. Na caixa de busca do topo, digite **`GAPDH`** e aperte Enter. *GAPDH* é um gene "de manutenção", ativo em quase todas as células.
5. Depois digite **`HOXD13`**. Os genes *HOX* comandam o desenvolvimento do embrião e ficam desligados em células do sangue adultas.

**Plano B** (se o IGV-Web não carregar as trilhas): <https://genome.ucsc.edu> → *Genome Browser* → **Human hg19** → digite `GAPDH` → abaixo da imagem, seção **Regulation** → **ENCODE Regulation** → ative as trilhas de H3K4Me3, H3K27Ac e DNase.

**Não feche o IGV-Web**: vamos voltar a ele no final da P2.

### 🤔 Registre suas observações

Dê dois cliques nesta célula para editá-la e preencha com **alto** ou **baixo**:

| Trilha | GAPDH | HOXD13 |
|---|---|---|
| H3K4me3 |  |  |
| H3K27ac |  |  |
| H3K27me3 |  |  |
| DNase (cromatina aberta) |  |  |

<details><summary><b>Ver resposta esperada</b></summary>

- **GAPDH:** H3K4me3 e H3K27ac **altos** no início do gene, DNase **aberta**, H3K27me3 **baixo** → gene **ativo**.
- **HOXD13:** um grande "tapete" de **H3K27me3** cobrindo a região e pouco sinal ativo → gene **reprimido por Polycomb**.

Confirme ou refute com o que **você** viu. É assim que a ciência funciona.
</details>

---
# P2 — Encontrando picos de ChIP-seq com o MACS3 (25 min)

### Relembrando a teoria
O **ChIP-seq** responde: *em que pontos do genoma está a proteína X?* Um anticorpo "pesca" os pedaços de DNA ligados à proteína; esses pedaços são sequenciados e alinhados ao genoma. Onde as leituras se **acumulam** acima do esperado, temos um **pico**.

O **controle (*input*)** é a mesma amostra processada **sem** o anticorpo. Ele mostra o ruído de fundo. Um pico verdadeiro precisa ser bem maior que o controle.

### A ferramenta: MACS3
**MACS** (*Model-based Analysis of ChIP-Seq*) é o programa mais usado no mundo para **chamar picos** (*peak calling*). É gratuito, de código aberto e escrito em Python. O MACS3 é a versão atual, sucessora do MACS2. Ele:
1. estima o tamanho médio dos fragmentos de DNA;
2. modela o ruído de fundo com uma distribuição estatística (Poisson) usando o controle;
3. testa cada região: "há mais leituras aqui do que o acaso explicaria?" → gera valor-p e valor-q.

### Os dados
Leituras de ChIP-seq da proteína **CTCF** e do respectivo controle, num trecho do **cromossomo 22 humano (hg19)**, com ~50 mil leituras cada. São os **dados de teste oficiais do projeto MACS**, derivados de dados públicos do ENCODE: pequenos o suficiente para rodar em segundos.

> **Quem é a CTCF?** Uma proteína que se liga a uma sequência específica do DNA e funciona como "isolador": organiza o genoma em alças e domínios. Por se ligar a um ponto preciso, gera **picos estreitos e nítidos**, ideais para aprender.

### Célula 1 — Instalar o MACS3
O `pip` é o instalador de pacotes do Python. O `-q` (*quiet*) deixa a saída mais limpa. Leva cerca de 1 minuto.

In [1]:
# Instala o MACS3 no computador do Google
!pip install -q macs3

# Confere se a instalação deu certo mostrando a versão
!macs3 --version

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.3/722.3 kB 3.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.2/190.2 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.3/405.3 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into accoun

### Célula 2 — Baixar os dados
Usamos o `wget`, um programa de terminal que baixa arquivos da internet. Os arquivos estão no repositório público do MACS no GitHub.

Os arquivos estão no formato **BED** e compactados (`.gz`). Cada linha é **uma leitura já alinhada ao genoma**: cromossomo, início, fim, ..., fita (+ ou −). Ou seja, o alinhamento (Bowtie2/BWA, Aula 2) já foi feito: começamos dele.

In [ ]:
# Endereço da pasta de dados de teste do MACS no GitHub
url = "https://raw.githubusercontent.com/macs3-project/MACS/master/test/"

# Baixa o arquivo de ChIP-seq (tratamento) e o de controle (input)
!wget -q {url}CTCF_SE_ChIP_chr22_50k.bed.gz
!wget -q {url}CTCF_SE_CTRL_chr22_50k.bed.gz

# zcat descompacta e mostra o conteúdo; head -5 mostra só as 5 primeiras linhas
print("Primeiras leituras do arquivo de ChIP-seq:")
!zcat CTCF_SE_ChIP_chr22_50k.bed.gz 2> /dev/null | head -5

# wc -l conta quantas linhas (= quantas leituras) há no arquivo
print("\nNúmero de leituras no ChIP-seq:")
!zcat CTCF_SE_ChIP_chr22_50k.bed.gz | wc -l
print("Número de leituras no controle:")
!zcat CTCF_SE_CTRL_chr22_50k.bed.gz | wc -l

🤔 **Quantas leituras há? Em que cromossomo elas estão?**

<details><summary><b>Ver resposta</b></summary>

Cerca de 50 mil leituras em cada arquivo (49.622 no ChIP-seq), todas no **chr22**. Um experimento real tem dezenas de milhões de leituras no genoma inteiro; aqui usamos uma amostra pequena para caber na aula.
</details>

### Célula 3 — Primeira tentativa de chamar os picos
Significado de cada opção do comando:

| Opção | Significado |
|---|---|
| `callpeak` | subcomando do MACS3 que chama picos |
| `-t` | arquivo de **tratamento** (ChIP-seq) |
| `-c` | arquivo de **controle** (input) |
| `-f BED` | formato dos arquivos |
| `-g hs` | tamanho do genoma: `hs` = *Homo sapiens* |
| `-n ctcf` | prefixo dos nomes dos arquivos de saída |
| `--outdir resultado` | pasta onde salvar os resultados |
| `-q 0.05` | só aceita picos com valor-q < 0,05 |

⚠️ **Esta célula vai mostrar um AVISO (WARNING) de propósito.** Leia a mensagem antes de seguir.

In [ ]:
# A barra invertida \ no fim da linha só serve para quebrar um comando longo em duas linhas
!macs3 callpeak -t CTCF_SE_ChIP_chr22_50k.bed.gz -c CTCF_SE_CTRL_chr22_50k.bed.gz \
    -f BED -g hs -n ctcf --outdir resultado -q 0.05

# Verifica se o arquivo de picos foi criado
import os
if os.path.exists("resultado/ctcf_peaks.narrowPeak"):
    print("\n>>> Arquivo de picos criado.")
else:
    print("\n>>> Nenhum arquivo de picos foi criado. Leia o WARNING acima!")

🤔 **O que o aviso diz? Qual solução ele mesmo sugere?**

<details><summary><b>Ver resposta</b></summary>

O MACS3 tenta **estimar sozinho o tamanho dos fragmentos de DNA** procurando pares de "montes" de leituras nas fitas + e −. Ele precisa de pelo menos **100 pares**, mas com só 50 mil leituras encontrou **0**. Sem essa estimativa, o processo para e **nenhum pico é gerado**.

A própria mensagem sugere a solução: usar `--nomodel` (não estimar) e `--extsize` (informar o tamanho manualmente).

**Lição para a vida real:** mensagens de aviso de programas de bioinformática quase sempre dizem o que está errado e como corrigir. Leia antes de pedir ajuda.
</details>

### Célula 4 — Chamar os picos informando o tamanho do fragmento
Acrescentamos duas opções:
- `--nomodel`: não tente estimar o tamanho do fragmento;
- `--extsize 200`: considere que cada fragmento tem ~200 pares de bases (valor típico de ChIP-seq).

In [ ]:
!macs3 callpeak -t CTCF_SE_ChIP_chr22_50k.bed.gz -c CTCF_SE_CTRL_chr22_50k.bed.gz \
    -f BED -g hs -n ctcf --outdir resultado -q 0.05 --nomodel --extsize 200

# ls -lh lista os arquivos da pasta, com o tamanho de cada um
print("\nArquivos gerados:")
!ls -lh resultado/

**Arquivos gerados pelo MACS3:**

| Arquivo | Conteúdo |
|---|---|
| `ctcf_peaks.narrowPeak` | **A lista de picos** (o mais importante) |
| `ctcf_summits.bed` | A posição exata do cume (ponto mais alto) de cada pico |
| `ctcf_peaks.xls` | A mesma lista em formato de tabela legível (apesar da extensão, é texto) |

### Célula 5 — Ler e explorar os picos
Vamos abrir o `.narrowPeak` com o **pandas**, a biblioteca de tabelas do Python (Aula 1). Uma *biblioteca* é um conjunto de ferramentas prontas que importamos com `import`.

O arquivo não tem cabeçalho, então damos nome às 10 colunas:

| Coluna | Nome que usaremos | Significado |
|---|---|---|
| 1–3 | cromossomo, inicio, fim | Onde está o pico |
| 4–6 | nome, pontuacao, fita | Identificação e exibição |
| 7 | **enriquecimento** | Quantas vezes o sinal é maior que o fundo |
| 8 | menos_log10_p | −log₁₀(valor-p) |
| 9 | **menos_log10_q** | −log₁₀(valor-q): **quanto maior, mais confiável** |
| 10 | cume | Posição do ponto mais alto, contada a partir do início |

> **Lendo o −log₁₀:** valor-q = 0,01 → 2; valor-q = 0,0001 → 4. **Número maior = mais significativo.**

In [ ]:
import pandas as pd   # "pd" é um apelido curto para a biblioteca pandas

# Nomes que daremos às 10 colunas do arquivo
colunas = ["cromossomo", "inicio", "fim", "nome", "pontuacao", "fita",
           "enriquecimento", "menos_log10_p", "menos_log10_q", "cume"]

# Lê o arquivo: sep="\t" indica que as colunas são separadas por tabulação
picos = pd.read_csv("resultado/ctcf_peaks.narrowPeak", sep="\t", names=colunas)

# Cria uma nova coluna: largura do pico = fim - início
picos["largura"] = picos["fim"] - picos["inicio"]

print("Número de picos encontrados:", len(picos))
print("Largura média dos picos (pb):", round(picos["largura"].mean()))
print("Largura mínima e máxima (pb):", picos["largura"].min(), "e", picos["largura"].max())

# Ordena do mais significativo para o menos e mostra os 5 primeiros
picos.sort_values("menos_log10_q", ascending=False).head(5)

> **Resultado de referência** (obtido no teste desta aula, MACS3 3.0.5): **802 picos**, largura média de **~390 pb**. O pico mais forte fica em **chr22:37.252.291–37.252.808**, com enriquecimento de ~58 vezes sobre o fundo.

### Célula 6 — Visualizar os picos
Usamos o **matplotlib**, a biblioteca de gráficos mais usada em Python. Faremos dois gráficos lado a lado:
- **Histograma** da largura dos picos: quantos picos têm cada largura.
- **Dispersão** enriquecimento × significância: cada ponto é um pico.

In [ ]:
import matplotlib.pyplot as plt

# Cria uma figura com 1 linha e 2 gráficos ("eixos")
fig, eixos = plt.subplots(1, 2, figsize=(12, 4))

# Gráfico 1: histograma das larguras (40 barras)
eixos[0].hist(picos["largura"], bins=40, color="steelblue")
eixos[0].set_title("Largura dos picos")
eixos[0].set_xlabel("Largura (pb)")
eixos[0].set_ylabel("Número de picos")

# Gráfico 2: cada ponto é um pico
eixos[1].scatter(picos["enriquecimento"], picos["menos_log10_q"], s=8, color="darkred")
eixos[1].set_title("Enriquecimento × significância")
eixos[1].set_xlabel("Enriquecimento sobre o controle (vezes)")
eixos[1].set_ylabel("-log10(valor-q)")

plt.tight_layout()   # ajusta espaçamentos para nada ficar sobreposto
plt.show()

🤔 **Os picos são estreitos (centenas de pb) ou largos (milhares)? Isso combina com uma proteína como a CTCF? Picos mais enriquecidos são também os mais significativos?**

<details><summary><b>Ver resposta</b></summary>

- São **estreitos** (centenas de pb). Combina com a CTCF, que se liga a um ponto preciso do DNA. Marcas como H3K27me3 dariam domínios largos, e aí usaríamos o modo `--broad` do MACS.
- Sim: a dispersão mostra uma tendência clara, maior enriquecimento acompanha maior −log₁₀(q). Mas não é uma linha perfeita, porque a significância também depende de quantas leituras há na região.
</details>

### Célula 7 — Ver seus picos no IGV-Web
Vamos baixar o arquivo de picos para o seu computador e abri-lo no IGV-Web, ao lado dos dados do ENCODE da P1.

In [ ]:
# Baixa o arquivo para o seu computador (funciona no Google Colab)
try:
    from google.colab import files
    files.download("resultado/ctcf_peaks.narrowPeak")
except ImportError:
    print("Fora do Colab: o arquivo está em resultado/ctcf_peaks.narrowPeak")

1. Volte à aba do **IGV-Web** (genoma **hg19**).
2. Menu **Tracks → Local File** → selecione `ctcf_peaks.narrowPeak`.
3. Na caixa de posição, digite `chr22:37,250,000-37,256,000` (região do pico mais forte).
4. Compare com a trilha de **CTCF do ENCODE** carregada na P1.

🤔 **Seus picos coincidem com os do ENCODE?**

> **Esse é o momento-chave da prática:** com uma ferramenta de código aberto e dados públicos, você acabou de reproduzir um resultado de um consórcio internacional.

---
# P3 — Metilação diferencial (25 min)

### Relembrando a teoria
No sequenciamento com **bissulfito**, citosinas **não metiladas** viram T e as **metiladas** continuam C. Para cada CpG:

> **% metilação = leituras com C ÷ (leituras com C + leituras com T) × 100**

O **Bismark** alinha as leituras e conta C e T em cada posição. O **methylKit** (pacote de R) faz a estatística: filtra, compara grupos e encontra os **CpGs diferencialmente metilados (DMCs)**.

### Os dados
Arquivos de exemplo **oficiais do methylKit**: 4 amostras (2 **teste** e 2 **controle**) de um trecho do **cromossomo 21 humano**. Cada linha é um CpG com:

| Coluna | Significado |
|---|---|
| `chrBase`, `chr`, `base` | Identificação e posição do CpG |
| `strand` | Fita do DNA |
| `coverage` | **Cobertura**: quantas leituras cobrem esse CpG |
| `freqC` | % de leituras com C (= metilado) |
| `freqT` | % de leituras com T (= não metilado) |

É exatamente o tipo de tabela que sai do Bismark.

> **Por que Python e não o methylKit?** Instalar o methylKit no Colab leva mais de 15 minutos. Faremos **os mesmos passos** com bibliotecas Python que já vêm no Colab. Em cada célula indicamos a função equivalente do methylKit. O script em R está no Apêndice A do roteiro, para casa.

### Célula 8 — Carregar as quatro amostras *(equivale a `methRead` no methylKit)*
O pandas consegue ler um arquivo direto de um endereço da internet. Para cada amostra, convertemos as porcentagens em **contagens de leituras**, que é do que o teste estatístico precisa.

Exemplo: cobertura 12 e freqC 25% → 3 leituras metiladas e 9 não metiladas.

In [ ]:
import pandas as pd
import numpy as np   # numpy: biblioteca de cálculo numérico

# Pasta com os dados de exemplo do methylKit no GitHub
base = "https://raw.githubusercontent.com/al2na/methylKit/master/inst/extdata/"

# Dicionário: nome que daremos à amostra -> nome do arquivo
amostras = {"teste1": "test1.myCpG.txt",
            "teste2": "test2.myCpG.txt",
            "controle1": "control1.myCpG.txt",
            "controle2": "control2.myCpG.txt"}

dados = {}   # aqui guardaremos as 4 tabelas
for nome, arquivo in amostras.items():          # repete para cada amostra
    df = pd.read_csv(base + arquivo, sep="\t")
    # Converte % em número de leituras metiladas (arredondando) e não metiladas
    df["metilados"] = (df["coverage"] * df["freqC"] / 100).round().astype(int)
    df["nao_metilados"] = df["coverage"] - df["metilados"]
    dados[nome] = df
    print(f"{nome:10s}: {len(df):5d} CpGs | cobertura mediana = {df.coverage.median():3.0f} "
          f"| metilação média = {df.freqC.mean():.1f}%")

# Mostra as 5 primeiras linhas da amostra teste1
dados["teste1"].head()

### Célula 9 — Distribuição da metilação *(equivale a `getMethylationStats`)*
Um histograma por amostra: quantos CpGs têm cada nível de metilação.

In [ ]:
import matplotlib.pyplot as plt

fig, eixos = plt.subplots(1, 4, figsize=(15, 3.5), sharey=True)  # 4 gráficos lado a lado
for eixo, (nome, df) in zip(eixos, dados.items()):
    eixo.hist(df["freqC"], bins=20, color="seagreen")
    eixo.set_title(nome)
    eixo.set_xlabel("% de metilação")
eixos[0].set_ylabel("Número de CpGs")
plt.tight_layout()
plt.show()

🤔 **Qual é a forma dos histogramas? Por que há tão poucos CpGs com metilação intermediária (~50%)?**

<details><summary><b>Ver resposta</b></summary>

Forma de **"U"** (bimodal): muitos CpGs perto de 0% e de 100%. Numa única célula, um CpG de uma cópia do DNA está **metilado ou não**: não existe "meio metilado". Valores intermediários aparecem quando a amostra mistura tipos celulares ou quando os dois alelos (do pai e da mãe) diferem.
</details>

### Célula 10 — Filtrar e unir as amostras *(equivale a `filterByCoverage`, `unite` e `getCorrelation`)*

1. **Filtro de cobertura:** CpGs com poucas leituras dão porcentagens pouco confiáveis (com 2 leituras, só existem 0%, 50% ou 100%). Mantemos só CpGs com **≥ 10 leituras**.
2. **União:** para comparar grupos, precisamos de CpGs presentes **nas 4 amostras**.
3. **Correlação:** medimos o quanto as amostras se parecem (1 = idênticas; 0 = sem relação).

In [ ]:
# 1. Filtro: mantém só as linhas com cobertura >= 10
filtrados = {nome: df[df["coverage"] >= 10] for nome, df in dados.items()}

# 2. União: junta as 4 tabelas, mantendo só os CpGs presentes em todas
tabela = None
for nome, df in filtrados.items():
    # renomeia as colunas para saber de qual amostra cada uma veio (ex.: met_teste1)
    parte = df[["chrBase", "chr", "base", "metilados", "nao_metilados", "freqC"]].rename(
        columns={"metilados": f"met_{nome}",
                 "nao_metilados": f"naomet_{nome}",
                 "freqC": f"pct_{nome}"})
    # merge = junção: só ficam os CpGs em comum
    tabela = parte if tabela is None else tabela.merge(parte, on=["chrBase", "chr", "base"])

print("CpGs presentes nas 4 amostras após o filtro:", len(tabela))

# 3. Matriz de correlação entre as porcentagens de metilação das amostras
colunas_pct = [c for c in tabela.columns if c.startswith("pct_")]
tabela[colunas_pct].corr().round(2)

🤔 **As réplicas do mesmo grupo (teste1 × teste2; controle1 × controle2) são mais parecidas entre si do que com o outro grupo?**

<details><summary><b>Ver resposta</b></summary>

Sim. Resultado de referência: **963 CpGs** em comum; correlação de **0,93** entre os testes e **0,95** entre os controles, contra **~0,88** entre grupos. Réplicas parecidas entre si e diferentes do outro grupo indicam boa qualidade experimental e sugerem uma diferença biológica real entre os grupos.
</details>

### Célula 11 — Teste estatístico *(equivale a `calculateDiffMeth`)*

Para **cada CpG**, perguntamos: *a proporção de leituras metiladas é diferente entre teste e controle?*

**Teste exato de Fisher**, em uma frase: montamos uma tabela 2×2 e calculamos a probabilidade de uma diferença tão grande surgir por acaso (o **valor-p**).

|  | Metiladas | Não metiladas |
|---|---|---|
| **Teste** | a | b |
| **Controle** | c | d |

**Correção para múltiplos testes:** fazemos ~1.000 testes ao mesmo tempo. Com valor-p < 0,05, esperaríamos ~50 falsos positivos só por acaso! O método de **Benjamini-Hochberg** transforma o valor-p em **valor-q**, que controla a **taxa de falsas descobertas (FDR)**. É a mesma lógica do DESeq2 da Aula 4.

> *Nota:* o methylKit usa, por padrão, **regressão logística**, que considera a variação entre réplicas. Nossa versão simplificada **soma as réplicas** de cada grupo. Os resultados são parecidos, mas não idênticos.

In [ ]:
from scipy.stats import fisher_exact                      # teste de Fisher
from statsmodels.stats.multitest import multipletests     # correção para múltiplos testes

t = tabela   # apelido curto

# Soma as leituras das duas réplicas de cada grupo
t["met_teste"]    = t["met_teste1"] + t["met_teste2"]
t["naomet_teste"] = t["naomet_teste1"] + t["naomet_teste2"]
t["met_ctrl"]     = t["met_controle1"] + t["met_controle2"]
t["naomet_ctrl"]  = t["naomet_controle1"] + t["naomet_controle2"]

# % de metilação de cada grupo e a diferença entre eles
t["pct_teste"] = 100 * t["met_teste"] / (t["met_teste"] + t["naomet_teste"])
t["pct_ctrl"]  = 100 * t["met_ctrl"]  / (t["met_ctrl"]  + t["naomet_ctrl"])
t["diferenca"] = t["pct_teste"] - t["pct_ctrl"]   # positiva = mais metilado no grupo teste

# Teste de Fisher em cada CpG; [1] pega o valor-p do resultado
t["p"] = [fisher_exact([[a, b], [c, d]])[1]
          for a, b, c, d in zip(t["met_teste"], t["naomet_teste"],
                                t["met_ctrl"], t["naomet_ctrl"])]

# Correção de Benjamini-Hochberg ("fdr_bh"): valor-p -> valor-q
t["q"] = multipletests(t["p"], method="fdr_bh")[1]

print("Teste concluído para", len(t), "CpGs")
print("CpGs com valor-p < 0,05:", (t["p"] < 0.05).sum())
print("CpGs com valor-q < 0,05:", (t["q"] < 0.05).sum())

### Célula 12 — CpGs diferencialmente metilados e gráfico de vulcão *(equivale a `getMethylDiff`)*

Critério usual para chamar um CpG de **diferencialmente metilado (DMC)**:
- diferença de metilação de **pelo menos 25 pontos percentuais** (relevância biológica), **e**
- **valor-q < 0,01** (confiança estatística).

O **gráfico de vulcão** mostra os dois critérios ao mesmo tempo: no eixo x, o tamanho da diferença; no eixo y, a significância. As linhas tracejadas marcam os limites.

In [ ]:
# Filtra os CpGs que cumprem os dois critérios (abs = valor absoluto, ignora o sinal)
dmc = t[(t["diferenca"].abs() >= 25) & (t["q"] < 0.01)]

print("CpGs diferencialmente metilados (|diferença| >= 25% e q < 0,01):", len(dmc))
print("  Hipermetilados no grupo teste:", (dmc["diferenca"] > 0).sum())
print("  Hipometilados no grupo teste: ", (dmc["diferenca"] < 0).sum())

# Gráfico de vulcão: DMCs em vermelho, o resto em cinza
plt.figure(figsize=(7, 5.5))
cor = np.where(t.index.isin(dmc.index), "crimson", "lightgray")
# somamos 1e-300 para evitar log de zero quando q é extremamente pequeno
plt.scatter(t["diferenca"], -np.log10(t["q"] + 1e-300), c=cor, s=10)
plt.axvline(25, ls="--", c="k"); plt.axvline(-25, ls="--", c="k")   # limites de diferença
plt.axhline(2, ls="--", c="k")                                        # q = 0,01 -> -log10 = 2
plt.xlabel("Diferença de metilação (teste − controle, pontos %)")
plt.ylabel("-log10(valor-q)")
plt.title("Gráfico de vulcão — metilação diferencial")
plt.show()

# Os 10 DMCs mais significativos
dmc.sort_values("q")[["chr", "base", "pct_teste", "pct_ctrl", "diferenca", "q"]].head(10)

> **Resultado de referência:** **127 DMCs** (72 hipermetilados e 55 hipometilados no grupo teste).

🤔 **O que significa um ponto no canto superior direito do vulcão? E no superior esquerdo? Por que a maioria dos pontos fica cinza perto do centro?**

<details><summary><b>Ver resposta</b></summary>

- **Superior direito:** CpG muito **mais metilado no grupo teste** (hipermetilado) e com alta confiança.
- **Superior esquerdo:** muito **menos metilado no grupo teste** (hipometilado), com alta confiança.
- **Centro, cinza:** a maioria dos CpGs **não muda** entre os grupos, ou muda pouco, ou a diferença não é estatisticamente confiável. É o esperado: um tratamento altera uma pequena fração do epigenoma.
</details>

---
# Para a lista de exercícios

Use as células abaixo. Elas já estão preparadas: basta mudar os números indicados e executar.

**Exercício P2 estendido:** rode o MACS3 com `-q 0.01` e depois com `--extsize 150`. Quantos picos você obtém em cada caso? Explique por que o número muda.

In [ ]:
# Mude os valores abaixo e execute
limite_q = 0.01      # experimente 0.05 e 0.01
tamanho  = 200       # experimente 200 e 150

!macs3 callpeak -t CTCF_SE_ChIP_chr22_50k.bed.gz -c CTCF_SE_CTRL_chr22_50k.bed.gz \
    -f BED -g hs -n teste --outdir exercicio -q {limite_q} --nomodel --extsize {tamanho} 2> /dev/null

# 2> /dev/null esconde as mensagens do MACS3; abaixo contamos os picos
n = len(pd.read_csv("exercicio/teste_peaks.narrowPeak", sep="\t", header=None))
print(f"q = {limite_q} e extsize = {tamanho}: {n} picos")

**Exercício P3 estendido:** refaça a análise exigindo cobertura mínima de **20** leituras. O número de DMCs aumentou ou diminuiu? Por quê?

In [ ]:
cobertura_minima = 20   # mude aqui (o valor usado na aula foi 10)

# Repete filtro + união + teste numa só função, para comparar facilmente
def contar_dmcs(cob):
    filt = {n: d[d["coverage"] >= cob] for n, d in dados.items()}
    tab = None
    for n, d in filt.items():
        parte = d[["chrBase", "metilados", "nao_metilados"]].rename(
            columns={"metilados": f"met_{n}", "nao_metilados": f"naomet_{n}"})
        tab = parte if tab is None else tab.merge(parte, on="chrBase")
    mt, nt = tab.met_teste1 + tab.met_teste2, tab.naomet_teste1 + tab.naomet_teste2
    mc, nc = tab.met_controle1 + tab.met_controle2, tab.naomet_controle1 + tab.naomet_controle2
    dif = 100 * mt / (mt + nt) - 100 * mc / (mc + nc)
    p = [fisher_exact([[a, b], [c, d]])[1] for a, b, c, d in zip(mt, nt, mc, nc)]
    q = multipletests(p, method="fdr_bh")[1]
    return len(tab), int(((dif.abs() >= 25) & (q < 0.01)).sum())

for cob in [10, cobertura_minima]:
    n_cpg, n_dmc = contar_dmcs(cob)
    print(f"Cobertura >= {cob:2d}: {n_cpg:4d} CpGs em comum | {n_dmc:3d} DMCs")

---
# Resumo: o que você fez hoje

| Prática | Conceito teórico | Ferramenta | O que você viu |
|---|---|---|---|
| **P1** | Marcas ativas × repressivas | IGV-Web + ENCODE | GAPDH ativo; HOXD reprimido por H3K27me3 |
| **P2** | ChIP-seq, controle, valor-q | MACS3 | ~800 picos estreitos de CTCF que coincidem com o ENCODE |
| **P3** | Bissulfito, cobertura, FDR | pandas + SciPy (lógica do methylKit) | Distribuição em "U", réplicas concordantes, ~127 DMCs |

**O fluxo completo da epigenômica:**
```
Amostra → técnica de laboratório (bissulfito, ChIP-seq, ATAC-seq) → sequenciamento → FASTQ
   → controle de qualidade (FastQC, Trimmomatic)            [Aula 2]
   → alinhamento (Bowtie2/BWA; Bismark para bissulfito)     [Aula 2 + hoje]
   → quantificação (MACS para picos; methylKit p/ metilação) [hoje]
   → visualização (IGV, UCSC) e integração com RNA-seq       [hoje + Aula 4]
```

**Documentação das ferramentas:** [MACS3](https://github.com/macs3-project/MACS) · [methylKit](https://bioconductor.org/packages/methylKit) · [Bismark](https://github.com/FelixKrueger/Bismark) · [IGV](https://igv.org) · [UCSC Genome Browser](https://genome.ucsc.edu) · [ENCODE](https://www.encodeproject.org)